# API call

In [26]:
import requests
import json
import os
import pandas as pd
from time import sleep
import glob

pd.set_option('display.max_columns', None)

In [4]:
all_matches = pd.read_csv("../data/raw_data/matches/pgl_wallachia/data.csv")

In [5]:
all_matches.head()

,match_id,start_time,win,hero_id,account_id,leaguename
0,8730786393,1773606284,True,74,312436974,PGL Wallachia 2026 Season 7
1,8730786393,1773606284,True,99,56351509,PGL Wallachia 2026 Season 7
2,8730786393,1773606284,True,64,93817671,PGL Wallachia 2026 Season 7
3,8730786393,1773606284,True,79,103735745,PGL Wallachia 2026 Season 7
4,8730786393,1773606284,False,37,16497807,PGL Wallachia 2026 Season 7


In [7]:
all_matches_ids = all_matches['match_id'].unique()

In [8]:
def fetch_match_data(match_id):
    url = f"https://api.opendota.com/api/matches/{match_id}"
    response = requests.get(url)

    if response.status_code == 200:
        match_data = response.json()
        return match_data
    else:
        print(f"Failed to fetch match data. Status code: {response.status_code}")
        return None

In [9]:
def inspect_match(match_data):
    if not match_data:
        print("No data to inspect.")
        return

    # Basic match info
    print("\n🏆 Match Summary:")
    print(f"Match ID: {match_data['match_id']}")
    print(f"Duration: {match_data['duration'] // 60} minutes")
    print(f"Radiant Win: {match_data['radiant_win']}")
    print(f"Game Mode: {match_data['game_mode']}")
    print(f"Start Time: {match_data['start_time']}")

    # Player stats
    print("\n👥 Player Stats:")
    for player in match_data['players']:
        print(f"Player: {player.get('personaname', 'Anonymous')}")
        print(f"  Hero ID: {player['hero_id']}")
        print(f"  K/D/A: {player['kills']}/{player['deaths']}/{player['assists']}")
        print(f"  GPM/XPM: {player['gold_per_min']}/{player['xp_per_min']}")
        print(f"  Last Hits: {player['last_hits']}")
        print(f"  Items: {[player.get(f'item_{i}') for i in range(6)]}")
        print("")

In [15]:
for match in all_matches_ids:
    match_data = fetch_match_data(match)
    with open(f"../data/raw_data/matches/pgl_wallachia/match_{match}.json", "w") as f:
        json.dump(match_data, f, indent=4)
    sleep(2)  # To avoid hitting API rate limits

In [5]:
match_data = fetch_match_data(8420814829)

In [8]:
match_data.keys()

dict_keys(['version', 'match_id', 'draft_timings', 'teamfights', 'objectives', 'chat', 'radiant_gold_adv', 'radiant_xp_adv', 'pauses', 'cosmetics', 'players', 'leagueid', 'start_time', 'duration', 'series_id', 'series_type', 'cluster', 'replay_salt', 'radiant_win', 'pre_game_duration', 'match_seq_num', 'tower_status_radiant', 'tower_status_dire', 'barracks_status_radiant', 'barracks_status_dire', 'first_blood_time', 'lobby_type', 'human_players', 'game_mode', 'flags', 'engine', 'radiant_score', 'dire_score', 'picks_bans', 'od_data', 'metadata', 'replay_url', 'patch', 'region', 'all_word_counts', 'my_word_counts', 'throw', 'loss'])

In [12]:
match_data['teamfights'][0]

{'start': 212,
 'end': 262,
 'last_death': 247,
 'deaths': 3,
 'players': [{'deaths_pos': {},
   'ability_uses': {'lich_frost_nova': 1},
   'ability_targets': {},
   'item_uses': {'ward_dispenser': 2, 'tango': 1},
   'killed': {},
   'deaths': 0,
   'buybacks': 0,
   'damage': 193,
   'healing': 0,
   'gold_delta': 0,
   'xp_delta': 138,
   'xp_start': 870,
   'xp_end': 1008},
  {'deaths_pos': {},
   'ability_uses': {},
   'ability_targets': {},
   'item_uses': {'tango': 1},
   'killed': {},
   'deaths': 0,
   'buybacks': 0,
   'damage': 51,
   'healing': 0,
   'gold_delta': 168,
   'xp_delta': 252,
   'xp_start': 1001,
   'xp_end': 1253},
  {'deaths_pos': {},
   'ability_uses': {'necrolyte_death_pulse': 4},
   'ability_targets': {},
   'item_uses': {'tango': 1},
   'killed': {'npc_dota_hero_witch_doctor': 1},
   'deaths': 0,
   'buybacks': 0,
   'damage': 957,
   'healing': 471,
   'gold_delta': 305,
   'xp_delta': 472,
   'xp_start': 862,
   'xp_end': 1334},
  {'deaths_pos': {'95.7':

## Assets pulling from API

In [16]:
heroes = requests.get("https://api.opendota.com/api/heroes")
with open("../data/raw_data/assets_data/heroes.json", "w") as f:
    json.dump(heroes.json(), f, indent=4)

## Teams and players info from all matches

In [24]:
LANE_ROLE_MAP = {1: "safe", 2: "mid", 3: "off"}

def infer_positions(team_players: list[dict]) -> dict[int, str]:
    """
    Infer pos1-pos5 role labels for a team's 5 players.
 
    Strategy:
      - Pos 2 (Mid):          sole player with lane_role=2, or highest net_worth among lane_role=2
      - Pos 1 (Carry):        remaining safe-lane player (lane_role=1) with higher net_worth
      - Pos 5 (Hard Support): remaining safe-lane player (lane_role=1) with lower net_worth
      - Pos 3 (Offlane):      off-lane player (lane_role=3 / roaming) with higher net_worth
      - Pos 4 (Soft Support): off-lane player (lane_role=3 / roaming) with lower net_worth
 
    Falls back to net_worth ranking across the whole team if lane data is ambiguous
    (e.g. lane_role is missing or all players have the same value).
 
    Returns: dict mapping account_id -> position label
    """
    POSITIONS = ["Carry", "Mid", "Offlane", "Soft Support", "Hard Support"]

    # Bucket by lane
    safe  = [p for p in team_players if p.get("lane_role") == 1]
    mid   = [p for p in team_players if p.get("lane_role") == 2]
    off   = [p for p in team_players if p.get("lane_role") == 3 or p.get("is_roaming")]


# Remove duplicates that appear in multiple buckets (is_roaming can overlap lane_role=3)
    off = list({p["account_id"]: p for p in off}.values())
 
    assigned: dict[int, str] = {}
 
    # --- Mid (pos 2) ---
    mid_sorted = sorted(mid, key=lambda p: p.get("net_worth", 0), reverse=True)
    if mid_sorted:
        assigned[mid_sorted[0]["account_id"]] = "Mid"
        # Any extra mid players (shouldn't happen but be safe) go to unassigned pool
        for p in mid_sorted[1:]:
            off.append(p)
 
    # --- Safe lane: Carry (pos 1) vs Hard Support (pos 5) ---
    # Remove already-assigned players
    safe_unassigned = [p for p in safe if p["account_id"] not in assigned]
    safe_sorted = sorted(safe_unassigned, key=lambda p: p.get("net_worth", 0), reverse=True)
    if len(safe_sorted) >= 2:
        assigned[safe_sorted[0]["account_id"]] = "Carry"
        assigned[safe_sorted[1]["account_id"]] = "Hard Support"
    elif len(safe_sorted) == 1:
        assigned[safe_sorted[0]["account_id"]] = "Carry"
 
    # --- Off lane: Offlane (pos 3) vs Soft Support (pos 4) ---
    off_unassigned = [p for p in off if p["account_id"] not in assigned]
    off_sorted = sorted(off_unassigned, key=lambda p: p.get("net_worth", 0), reverse=True)
    if len(off_sorted) >= 2:
        assigned[off_sorted[0]["account_id"]] = "Offlane"
        assigned[off_sorted[1]["account_id"]] = "Soft Support"
    elif len(off_sorted) == 1:
        assigned[off_sorted[0]["account_id"]] = "Offlane"
 
    # --- Fallback: any player not yet assigned gets ranked by net_worth ---
    unassigned = [p for p in team_players if p["account_id"] not in assigned]
    if unassigned:
        used_roles = set(assigned.values())
        remaining_roles = [r for r in POSITIONS if r not in used_roles]
        unassigned_sorted = sorted(unassigned, key=lambda p: p.get("net_worth", 0), reverse=True)
        for p, role in zip(unassigned_sorted, remaining_roles):
            assigned[p["account_id"]] = role
 
    return assigned


In [25]:
def extract_from_match(file_path: str):
    """
    Returns (team_rows, player_rows) for one match file.
    Returns ([], []) if the file lacks team data (e.g. public matchmaking).
    """
    with open(file_path) as f:
        data = json.load(f)
 
    # Skip public games with no team metadata
    if not data.get("radiant_team") or not data.get("dire_team"):
        return [], []
 
    match_id   = data["match_id"]
    all_players = data["players"]
 
    team_rows   = []
    player_rows = []
 
    sides = [
        ("radiant", data["radiant_team"], 0),
        ("dire",    data["dire_team"],    1),
    ]
 
    for side, team_data, team_number in sides:
        team_id   = team_data["team_id"]
        team_name = team_data["name"]
        team_tag  = team_data.get("tag")
 
        team_players = [p for p in all_players if p["team_number"] == team_number]
 
        # Infer positions
        position_map = infer_positions(team_players)  # account_id -> role label
 
        # Sort by inferred position for slot columns
        role_order = ["Carry", "Mid", "Offlane", "Soft Support", "Hard Support"]
        players_with_role = sorted(
            team_players,
            key=lambda p: role_order.index(position_map.get(p["account_id"], "Carry"))
        )
 
        # --- Teams row ---
        team_row = {
            "team_id":             team_id,
            "team_name":           team_name,
            "team_tag":            team_tag,
            "carry_player_id":     players_with_role[0]["account_id"] if len(players_with_role) > 0 else None,
            "mid_player_id":       players_with_role[1]["account_id"] if len(players_with_role) > 1 else None,
            "offlane_player_id":   players_with_role[2]["account_id"] if len(players_with_role) > 2 else None,
            "softsup_player_id":   players_with_role[3]["account_id"] if len(players_with_role) > 3 else None,
            "hardsup_player_id":   players_with_role[4]["account_id"] if len(players_with_role) > 4 else None,
        }
        team_rows.append(team_row)
 
        # --- Players rows ---
        for p in team_players:
            player_rows.append({
                "player_id":   p["account_id"],
                "team_id":     team_id,
                "name":        p.get("name"),        # pro tag (e.g. "skem")
                "personaname": p.get("personaname"), # Steam display name
                "position":    position_map.get(p["account_id"]),
            })
 
    return team_rows, player_rows

In [66]:
def extract_all(folder_path: str, output_dir: str = "."):
    teams = {}
    players = {}
 
    files = glob.glob(os.path.join(folder_path, "match_*.json"))
    print(f"Found {len(files)} match files...")
 
    skipped = 0
    for fp in files:
        team_rows, player_rows = extract_from_match(fp)
        if not team_rows:
            skipped += 1
            continue
        for row in team_rows:
            teams[row["team_id"]] = row      # later matches overwrite earlier ones

        for row in player_rows:
            players[row["player_id"]] = row 
 
    print(f"Skipped {skipped} public/incomplete matches")
 
    # --- Teams table ---
    # A team's roster can change over time; keep the most recent match seen
    df_teams = pd.DataFrame([team for team in teams.values()]).drop_duplicates(subset="team_id", keep="last").reset_index(drop=True)
 
    # --- Players table ---
    # A player can switch teams; keep most recent record per player
    df_players = pd.DataFrame([player for player in players.values()]).drop_duplicates(subset="player_id", keep="last").reset_index(drop=True)

    # Output
    teams_path   = os.path.join(output_dir, "teams.parquet")
    players_path = os.path.join(output_dir, "players.parquet")

    df_players = df_players[df_players["player_id"] != 103812784]

    df_teams.to_parquet(teams_path, index=False, engine="fastparquet")
    df_players.to_parquet(players_path, index=False, engine="fastparquet")
 
    print(f"Teams:   {len(df_teams)} rows  → {teams_path}")
    print(f"Players: {len(df_players)} rows  → {players_path}")
 
    return df_teams, df_players

In [50]:
sample_json = {'team_id': 8261500, 'team_name': 'Xtreme Gaming', 'team_tag': 'XG', 'carry_player_id': 898754153, 'mid_player_id': 173978074, 'offlane_player_id': 129958758, 'softsup_player_id': 101695162, 'hardsup_player_id': 94296097}
sample_df = pd.DataFrame([sample_json])

In [51]:
sample_df.head()

,team_id,team_name,team_tag,carry_player_id,mid_player_id,offlane_player_id,softsup_player_id,hardsup_player_id
0,8261500,Xtreme Gaming,XG,898754153,173978074,129958758,101695162,94296097


In [67]:
FOLDER = "../data/raw_data/matches/pgl_wallachia"
OUTPUT = "../data/raw_data/assets_data"

df_teams, df_players = extract_all(FOLDER, OUTPUT)

print("\n--- Teams sample ---")
print(df_teams.head())
print("\n--- Players sample ---")
print(df_players.head())

Found 124 match files...
Skipped 0 public/incomplete matches
Teams:   16 rows  → ../data/raw_data/assets_data/teams.parquet
Players: 80 rows  → ../data/raw_data/assets_data/players.parquet

--- Teams sample ---
   team_id      team_name team_tag  carry_player_id  mid_player_id  \
0  8261500  Xtreme Gaming       XG        898754153      173978074   
1  8255888   BetBoom Team  BetBoom        172099728      480412663   
2   726228    Vici Gaming       VG        320252024      137129583   
3  2586976             OG       OG        355168766      324277900   
4  9467224  Aurora Gaming   Aurora        152455523      301750126   

   offlane_player_id  softsup_player_id  hardsup_player_id  
0          129958758          101695162           94296097  
1          165564598          317880638          196878136  
2          118134220          150961567          111114687  
3          146711951          155494381          100594231  
4          126842529          256156323          320219866  

-

In [41]:
teams['8261500']

,8261500,8255888,726228,2586976,9467224,9823272,2163,7119388,8291895,36,9572001,9303484,9247354,2576071,9338413,9691969
team_id,8261500,8255888,726228,2586976,9467224,9823272,2163,7119388,8291895,36,9572001,9303484,9247354,2576071,9338413,9691969
team_name,Xtreme Gaming,BetBoom Team,Vici Gaming,OG,Aurora Gaming,Team Yandex,Team Liquid,Team Spirit,Tundra Esports,Natus Vincere,PARIVISION,HEROIC,Team Falcons,Yellow Submarine,MOUZ,Team Nemesis
team_tag,XG,BetBoom,VG,OG,Aurora,TY,Liquid,TSpirit,Tundra,NAVI,PV,HEROIC,FLCN,YeS,MOUZ,NMSS
carry_player_id,898754153,172099728,320252024,355168766,152455523,171262902,152962063,321580662,1171243748,957204049,1044002267,177203952,100058342,164506534,127617979,375507918
mid_player_id,173978074,480412663,137129583,324277900,301750126,312436974,201358612,106305042,93618577,185590374,106573901,1026694469,210053851,116865891,116585378,104512126


In [23]:
df.head()

,version,match_id,teamfights,pauses,objectives,chat,radiant_gold_adv,radiant_xp_adv,draft_timings,players,leagueid,start_time,duration,series_id,series_type,cluster,replay_salt,radiant_win,pre_game_duration,match_seq_num,tower_status_radiant,tower_status_dire,barracks_status_radiant,barracks_status_dire,first_blood_time,lobby_type,human_players,game_mode,flags,engine,radiant_score,dire_score,radiant_team_id,radiant_name,radiant_logo,radiant_team_complete,dire_team_id,dire_name,dire_logo,dire_team_complete,radiant_captain,dire_captain,picks_bans,metadata,replay_url,patch,region,comeback,stomp,od_data.has_api,od_data.has_gcdata,od_data.has_parsed,od_data.has_archive,league.leagueid,league.ticket,league.banner,league.tier,league.name,radiant_team.team_id,radiant_team.name,radiant_team.tag,radiant_team.logo_url,dire_team.team_id,dire_team.name,dire_team.tag,dire_team.logo_url,all_word_counts.glhf,all_word_counts.glgl,all_word_counts.gl,all_word_counts.g,all_word_counts.gg,all_word_counts.gygy,source_file,throw,loss,all_word_counts.hey,all_word_counts.teams,all_word_counts.as,all_word_counts.soon,all_word_counts.the,all_word_counts.draft,all_word_counts.is,all_word_counts.over,all_word_counts.please,all_word_counts.pause,all_word_counts.game,all_word_counts.ok,all_word_counts.ready,all_word_counts.r,all_word_counts.rdy,all_word_counts.unpause,all_word_counts.hf,all_word_counts.can,all_word_counts.dim,all_word_counts.lights,all_word_counts.abit,all_word_counts.hfhf,all_word_counts.sry,all_word_counts.d,all_word_counts.xd,all_word_counts.ggwp,all_word_counts.cimahi,all_word_counts.pride,all_word_counts.match,all_word_counts.beras,all_word_counts.dimoon,all_word_counts.ti,all_word_counts.ne,all_word_counts.v,all_word_counts.clube,all_word_counts.green,all_word_counts.bb,all_word_counts.and,all_word_counts.do,all_word_counts.not,all_word_counts.until,all_word_counts.further,all_word_counts.instructions,all_word_counts.thanks,all_word_counts.okl,all_word_counts.mode,all_word_counts.zayacw,all_word_counts.show,all_word_counts.me,all_word_counts.ill,all_word_counts.another,all_word_counts.hero,all_word_counts.wp,all_word_counts.pls,all_word_counts.no,all_word_counts.cliff,all_word_counts.lion,all_word_counts.in,all_word_counts.a,all_word_counts.cage,all_word_counts.try,all_word_counts.gwr,all_word_counts.mic,all_word_counts.niku,all_word_counts.kez,all_word_counts.ts,all_word_counts.voice,all_word_counts.vgyang,all_word_counts.sound,all_word_counts.crash,all_word_counts.ka,all_word_counts.le,all_word_counts.k,all_word_counts.ale,all_word_counts.kale,all_word_counts.lag,all_word_counts.we,all_word_counts.lagged,all_word_counts.god,all_word_counts.guys,all_word_counts.sec,all_word_counts.headset,all_word_counts.secx,all_word_counts.martin,all_word_counts.mouse,all_word_counts.packet,all_word_counts.loss,all_word_counts.oliver,all_word_counts.go,all_word_counts.let,all_word_counts.more,all_word_counts.mars,all_word_counts.noooooooooooooo,all_word_counts.cant,all_word_counts.best,all_word_counts.get,all_word_counts.it,all_word_counts.games,all_word_counts.row,all_word_counts.hotkey,all_word_counts.same,all_word_counts.arsenal,all_word_counts.-,all_word_counts.atleast,all_word_counts.yeah,all_word_counts.fa,all_word_counts.cup,all_word_counts.tho,all_word_counts.against,all_word_counts.div,all_word_counts.n,all_word_counts.packetloss,all_word_counts.broski,all_word_counts.ogxnova,all_word_counts.haha,all_word_counts.keyboard,all_word_counts.pogi,all_word_counts.guapo,all_word_counts.glglglgl,all_word_counts.hfh,all_word_counts.waiting,all_word_counts.for,all_word_counts.admins,all_word_counts.la,all_word_counts.gege,all_word_counts.its,all_word_counts.nd,all_word_counts.time,all_word_counts.i,all_word_counts.new,all_word_counts.bat,all_word_counts.already,all_word_counts.bydy,all_word_counts.ubivat,all_word_counts.ili,all_word_counts.bit,all_word_counts.pox,all_word_counts.hff,all_word_counts.lo,all_word_counts.siento,all_word_counts.wilson,al

In [1]:
import sys
sys.path.append("/Users/gjorgibogatinov/projects/random/dota2-analytics")

from src.utils.db_connection import DotaDB

In [5]:
import pandas as pd

heroes = pd.read_json("../data/raw_data/assets_data/heroes.json")
teams = pd.read_parquet("../data/raw_data/assets_data/teams.parquet")
players = pd.read_parquet("../data/raw_data/assets_data/players.parquet")

In [7]:
teams.drop(columns=["carry_player_id", "mid_player_id", "offlane_player_id", "softsup_player_id", "hardsup_player_id"], inplace=True)

In [11]:
players.drop(columns=["PERSONANAME"], inplace=True)

In [2]:
import os

os.environ["SNOWFLAKE_ACCOUNT"] = "bgvkhjf-qy29148"
os.environ["SNOWFLAKE_USER"] = "codechem"
os.environ["SNOWFLAKE_PASSWORD"] = "SjVi82pcPp8skb"
os.environ["SNOWFLAKE_WAREHOUSE"] = "dota2_wh"
os.environ["SNOWFLAKE_DATABASE"] = "dota2_db"
os.environ["SNOWFLAKE_SCHEMA"] = "dota2_schema"

In [3]:
import json

folder = "../data/raw_data/matches/pgl_wallachia"
matches = []
for file in os.listdir(folder):
    if file.endswith(".json"):
        with open(os.path.join(folder, file), "r") as f:
            matches.append(json.load(f))

In [4]:
import pandas as pd

matches_df = pd.DataFrame(matches)

In [5]:
with DotaDB() as db:
    db.insert_ignore(matches_df, "matches_raw", conflict_columns=["match_id"])

In [14]:
matches_df.columns

Index(['VERSION', 'MATCH_ID', 'TEAMFIGHTS', 'PAUSES', 'OBJECTIVES', 'CHAT',
       'RADIANT_GOLD_ADV', 'RADIANT_XP_ADV', 'COSMETICS', 'DRAFT_TIMINGS',
       'PLAYERS', 'LEAGUEID', 'START_TIME', 'DURATION', 'SERIES_ID',
       'SERIES_TYPE', 'CLUSTER', 'REPLAY_SALT', 'RADIANT_WIN',
       'PRE_GAME_DURATION', 'MATCH_SEQ_NUM', 'TOWER_STATUS_RADIANT',
       'TOWER_STATUS_DIRE', 'BARRACKS_STATUS_RADIANT', 'BARRACKS_STATUS_DIRE',
       'FIRST_BLOOD_TIME', 'LOBBY_TYPE', 'HUMAN_PLAYERS', 'GAME_MODE', 'FLAGS',
       'ENGINE', 'RADIANT_SCORE', 'DIRE_SCORE', 'RADIANT_TEAM_ID',
       'RADIANT_NAME', 'RADIANT_LOGO', 'RADIANT_TEAM_COMPLETE', 'DIRE_TEAM_ID',
       'DIRE_NAME', 'DIRE_LOGO', 'DIRE_TEAM_COMPLETE', 'RADIANT_CAPTAIN',
       'DIRE_CAPTAIN', 'PICKS_BANS', 'OD_DATA', 'LEAGUE', 'RADIANT_TEAM',
       'DIRE_TEAM', 'METADATA', 'REPLAY_URL', 'PATCH', 'REGION',
       'ALL_WORD_COUNTS', 'MY_WORD_COUNTS', 'COMEBACK', 'STOMP', 'THROW',
       'LOSS'],
      dtype='object')